# 06 - RQ2: So sánh 6 bộ phân loại học máy cổ điển trên nhóm đặc trưng `ghep` (S4-03 & S4-04)

**Bài tập lớn môn Học máy:** Phân loại 4 nhóm ảnh X-quang ngực (`COVID`, `Lung_Opacity`, `Normal`, `Viral Pneumonia`).  
**Câu hỏi nghiên cứu 2 (RQ2):** Trên nhóm đặc trưng thủ công tốt nhất đã chốt ở RQ1 (`ghep`: 152 chiều = Histogram độ xám 64 chiều + GLCM 60 chiều + LBP 28 chiều), bộ phân loại học máy cổ điển nào (`lr`: Logistic Regression, `knn`: k-NN, `dt`: Decision Tree, `rf`: Random Forest, `svm`: SVM nhân RBF, `gb`: Gradient Boosting) đạt Macro-F1 validation cao nhất?  
**Người phụ trách (Code Owner):** Nguyễn Minh Thuận (duyệt Lê Minh — S4-03) & Cả nhóm (S4-04).

## Nội dung thực hiện theo `results/threshold_v3.md`:
1. **Kiểm toán 12 run (`results/rq2_ghep_<clf>_s<seed>`):** Xác minh các cổng hợp lệ H1–H7, thiết lập 5-fold `StratifiedGroupKFold` (`group_id`), trọng số lớp (H3), chuẩn hoá `StandardScaler` chỉ fit trên train (H6), và đối chiếu mốc nền `B0 = 0,1627`, `A0 = 0,2397` (H4).
2. **Thống kê ngân sách tính toán và siêu tham số (H3, H5):** Đối chiếu siêu tham số chọn qua 5-fold CV, thời gian trích đặc trưng và thời gian huấn luyện từ `results/summary_rq2_full.json` (đã đóng băng trong `results/rq2_cost_rank_record.md` trước khi mở điểm).
3. **Xuất và trình bày bộ báo cáo bắt buộc R1, R2, R4, R5 (`src/bootstrap.py`):**
   - **R1:** Macro-F1 và Precision/Recall/F1 từng lớp (trung bình ± độ lệch chuẩn mẫu qua 3 seed đối với mô hình ngẫu nhiên `dt, rf, gb`; ghi rõ 1 lần chạy và không có độ lệch chuẩn đối với mô hình tất định `lr, knn, svm` theo Cổng H3) kèm khoảng tin cậy 95% bootstrap (2.000 lần).
   - **Mục 5 (Phép so sánh chính):** Kiểm định bootstrap ghép cặp 2.000 lần giữa ứng viên tốt nhất (`svm`) và tham chiếu `rf`.
   - **R2, R4, R5:** Ma trận con 2×2 `Normal – Lung_Opacity`, sai số chuẩn lấy mẫu xấp xỉ trên toàn tập validation ($n = 3.174$) và lớp `Viral Pneumonia` ($n = 203$), cùng các giới hạn phương pháp luận.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# Định vị thư mục gốc của repository
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "results" / "threshold_v3.md").exists() and (REPO_ROOT.parent / "results" / "threshold_v3.md").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.bootstrap import (
    align_predictions,
    build_rq2_report,
    load_predictions,
    load_rq2_predictions,
    macro_f1_from_labels,
    paired_bootstrap_diff,
)
from src.metrics import DEFAULT_CLASS_NAMES, compute_metrics

RESULTS_DIR = REPO_ROOT / "results"
REPORT_DIR = RESULTS_DIR / "report_rq2"
CLASSIFIERS_ORDER = ["lr", "knn", "dt", "rf", "svm", "gb"]
DETERMINISTIC_CLFS = ("lr", "knn", "svm")
SEEDS_ORDER = [42, 43, 44]
B0_MACRO_F1 = 0.1627
A0_BEST_MACRO_F1 = 0.2397

print(f"Thư mục gốc dự án: {REPO_ROOT}")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")


Thư mục gốc dự án: D:\Use Antigravity\Medical Vision
NumPy: 2.5.4 | Pandas: 2.2.3


## 1. Kiểm toán 12 Run thực nghiệm RQ2 và Cổng hợp lệ H1–H6

Mỗi run trong `results/rq2_ghep_<clf>_s<seed>` được kiểm tra:
- Tồn tại đủ 4 tệp `config.yaml`, `metrics.json`, `log`, `preds_val.csv` (Cổng H1).
- `allow_test = False`, `n_train = 14.815`, `n_val = 3.174`, mã băm `split_v1.csv` khớp chuẩn (Cổng H2, H6).
- Cross-validation `StratifiedGroupKFold` 5 fold theo `group_id` (`shuffle=True, random_state=42`), chuẩn hoá `StandardScaler` chỉ fit trên train cho `lr, knn, svm`, và thiết lập trọng số lớp đúng quy định Cổng H3 (`class_weight="balanced"` cho `lr, dt, svm`; `class_weight="balanced_subsample"` cho `rf`; `sample_weight="balanced"` cho `gb`; `knn` không hỗ trợ `class_weight`).
- Điểm `macro_f1` tính lại trực tiếp từ `preds_val.csv` khớp với `metrics.json` và vượt qua `B0 = 0,1627`, `A0 = 0,2397` (Cổng H4).


In [2]:
EXPECTED_COMMIT = "1a0925dcbe5e7a847c65f9654574806122f75b3d"
EXPECTED_SPLIT_SHA256 = "f99e03d3d5cfc32f2bbc9befbdef45a07a256458a837a6d420a6658ed7583d98"

audit_rows = []
for clf in CLASSIFIERS_ORDER:
    clf_seeds = [SEEDS_ORDER[0]] if clf in DETERMINISTIC_CLFS else SEEDS_ORDER
    for seed in clf_seeds:
        run_id = f"rq2_ghep_{clf}_s{seed}"
        run_dir = RESULTS_DIR / run_id
        for req_file in ("config.yaml", "metrics.json", "log", "preds_val.csv"):
            assert (run_dir / req_file).is_file(), f"Thiếu {req_file} trong {run_id}"

        cfg = yaml.safe_load((run_dir / "config.yaml").read_text(encoding="utf-8"))
        met = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
        preds_df = pd.read_csv(run_dir / "preds_val.csv")

        recomp = compute_metrics(preds_df["y_true"].to_numpy(), preds_df["y_pred"].to_numpy())
        exact_f1 = macro_f1_from_labels(preds_df["y_true"].to_numpy(), preds_df["y_pred"].to_numpy())

        assert cfg["commit"] == EXPECTED_COMMIT
        assert cfg["features"] == "ghep"
        assert cfg["split"]["sha256"] == EXPECTED_SPLIT_SHA256
        assert cfg["evaluation"]["allow_test"] is False
        assert cfg["cv"]["scheme"] == "StratifiedGroupKFold"
        assert cfg["cv"]["n_splits"] == 5 and cfg["cv"]["group_column"] == "group_id"
        if clf in DETERMINISTIC_CLFS:
            assert cfg["preprocessing"]["scaler"] == "StandardScaler"
            assert cfg["preprocessing"]["fit_on"] == "train_only"
        assert abs(recomp["macro_f1"] - met["macro_f1"]) < 1e-12
        assert abs(recomp["accuracy"] - met["accuracy"]) < 1e-12
        assert recomp["confusion_matrix"] == met["confusion_matrix"]
        assert exact_f1 > B0_MACRO_F1 and exact_f1 > A0_BEST_MACRO_F1

        sec = met["seconds"]
        audit_rows.append({
            "run_id": run_id,
            "classifier": clf,
            "seed": seed,
            "is_deterministic": cfg["deterministic"],
            "cv_best_macro_f1": met["cv_best_macro_f1"],
            "macro_f1_val": exact_f1,
            "accuracy_val": met["accuracy"],
            "cv_cpu_s": sec["cv_cpu"],
            "fit_cpu_s": sec["fit_cpu"],
            "predict_wall_s": sec["predict_wall"],
        })

audit_df = pd.DataFrame(audit_rows)
print("Kiểm toán 12/12 run RQ2: ĐẠT TUYỆT ĐỐI (H1-H6).")
print(audit_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


Kiểm toán 12/12 run RQ2: ĐẠT TUYỆT ĐỐI (H1-H6).
          run_id classifier  seed  is_deterministic  cv_best_macro_f1  macro_f1_val  accuracy_val  cv_cpu_s  fit_cpu_s  predict_wall_s
 rq2_ghep_lr_s42         lr    42              True            0.8335        0.8370        0.8302   86.8690   185.6380          0.0060
rq2_ghep_knn_s42        knn    42              True            0.8259        0.8242        0.8258   18.4670     0.0390          0.3970
 rq2_ghep_dt_s42         dt    42             False            0.7287        0.7255        0.7379  189.6010     4.3160          0.0020
 rq2_ghep_dt_s43         dt    43             False            0.7287        0.7289        0.7404  189.6010     4.2200          0.0020
 rq2_ghep_dt_s44         dt    44             False            0.7287        0.7269        0.7382  189.6010     4.2250          0.0020
 rq2_ghep_rf_s42         rf    42             False            0.8550        0.8589        0.8554 3086.9710   151.5550          0.1290
 rq2_gh

## 2. Ngân sách thời gian huấn luyện và siêu tham số chọn qua 5-fold CV (Cổng H3 & H5)

Tổng hợp siêu tham số tối ưu trên 5-fold `StratifiedGroupKFold` (`group_id`) và tổng thời gian CPU huấn luyện (`total_train_cpu_seconds` = giây CPU 5-fold CV + tổng giây CPU fit trên toàn bộ tập `train`) từ `results/summary_rq2_full.json` (đã đóng băng trong `results/rq2_cost_rank_record.md`).


In [3]:
summary_rq2 = json.loads((RESULTS_DIR / "summary_rq2_full.json").read_text(encoding="utf-8"))

cost_rows = []
for clf in CLASSIFIERS_ORDER:
    sub_clf = audit_df[audit_df["classifier"] == clf]
    cost_rows.append({
        "classifier": clf,
        "is_deterministic": clf in DETERMINISTIC_CLFS,
        "n_runs": len(sub_clf),
        "selected_hyperparameters": json.dumps(summary_rq2["selected_hyperparameters"][clf]),
        "cv_best_macro_f1": float(sub_clf["cv_best_macro_f1"].iloc[0]),
        "cv_cpu_s": summary_rq2["cv_cpu_seconds"][clf],
        "fit_cpu_mean_s": summary_rq2["fit_cpu_seconds_mean"][clf],
        "total_train_cpu_s": summary_rq2["total_train_cpu_seconds"][clf],
        "cost_rank": summary_rq2["cost_rank"].index(clf) + 1,
    })

cost_df = pd.DataFrame(cost_rows).sort_values("cost_rank")
print(
    f"Thời gian trích đặc trưng ghep (dùng chung): "
    f"train={summary_rq2['extract_cpu_seconds']['train']['ghep']:.2f}s, "
    f"val={summary_rq2['extract_cpu_seconds']['val']['ghep']:.2f}s "
    f"({summary_rq2['extract_cpu_seconds_per_image']['ghep']:.6f}s/ảnh)"
)
print("=== BẢNG NGÂN SÁCH HUẤN LUYỆN VÀ SIÊU THAM SỐ 5-FOLD CV (THỨ HẠNG CHI PHÍ TỪ RẺ ĐẾN ĐẮT) ===")
print(cost_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


Thời gian trích đặc trưng ghep (dùng chung): train=1155.27s, val=247.59s (0.077985s/ảnh)
=== BẢNG NGÂN SÁCH HUẤN LUYỆN VÀ SIÊU THAM SỐ 5-FOLD CV (THỨ HẠNG CHI PHÍ TỪ RẺ ĐẾN ĐẮT) ===
classifier  is_deterministic  n_runs                                    selected_hyperparameters  cv_best_macro_f1  cv_cpu_s  fit_cpu_mean_s  total_train_cpu_s  cost_rank
       knn              True       1                   {"n_neighbors": 5, "weights": "distance"}            0.8259   18.4670          0.0390            18.5060          1
        dt             False       3                    {"max_depth": 10, "min_samples_leaf": 1}            0.7287  189.6010          4.2540           202.3620          2
        lr              True       1                                                 {"C": 10.0}            0.8335   86.8690        185.6380           272.5070          3
       svm              True       1                               {"C": 10.0, "gamma": "scale"}            0.8907  419.0940         6

## 3. Báo cáo bắt buộc R1 & Phép so sánh chính của RQ2 (`src/bootstrap.py`)

- Gọi `build_rq2_report` để tạo/cập nhật 8 tệp báo cáo chuẩn tại `results/report_rq2/`.
- Theo Cổng H3: 3 bộ phân loại tất định (`lr, knn, svm`) chạy 1 lần (`n_seeds = 1`) và **không có độ lệch chuẩn** (`SD = N/A (tất định)` khi trình bày báo cáo; cột số `macro_f1_std = 0.0` trong CSV); 3 bộ phân loại ngẫu nhiên (`dt, rf, gb`) báo cáo trung bình ± độ lệch chuẩn mẫu qua 3 seed (`42, 43, 44`).
- Thực hiện phép so sánh chính của RQ2 (Mục 5 `results/threshold_v3.md`): bộ phân loại ứng viên tốt nhất (`svm`) so với tham chiếu `rf` bằng bootstrap ghép cặp 2.000 lần (`rng_seed = 42`), đồng thời đối chiếu chéo với `rq1_ghep_rf` (`500` cây).


In [4]:
report_paths = build_rq2_report(
    results_dir=RESULTS_DIR,
    out_dir=REPORT_DIR,
    features="ghep",
    classifiers=CLASSIFIERS_ORDER,
    seeds=SEEDS_ORDER,
    deterministic_classifiers=list(DETERMINISTIC_CLFS),
    reference="rf",
    n_boot=2000,
    rng_seed=42,
    expected_n=3174,
    threshold_path=RESULTS_DIR / "threshold_v3.md",
)

r1_summary_df = pd.read_csv(report_paths["r1_summary.csv"])
r1_boot_df = pd.read_csv(report_paths["r1_bootstrap_vs_reference.csv"])

r1_display = r1_summary_df.copy()
r1_display["macro_f1_report"] = r1_display.apply(
    lambda r: f"{r['macro_f1_mean']:.4f} (tất định, 1 run)"
    if bool(r["is_deterministic"])
    else f"{r['macro_f1_mean']:.4f} ± {r['macro_f1_std']:.4f} (3 seeds)",
    axis=1,
)
display_r1_cols = [
    "classifier",
    "macro_f1_report",
    "ci95_low",
    "ci95_high",
    "recall_COVID_mean",
    "recall_Lung_Opacity_mean",
    "recall_Normal_mean",
    "recall_Viral Pneumonia_mean",
]
print("=== BẢNG R1: MACRO-F1 VÀ RECALL TỪNG LỚP CỦA 6 BỘ PHÂN LOẠI TRÊN NHÓM GHEP ===")
print(r1_display.sort_values("macro_f1_mean", ascending=False)[display_r1_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== KIỂM ĐỊNH BOOTSTRAP GHÉP CẶP 2.000 LẦN SO VỚI THAM CHIẾU RF (MỤC 5 THRESHOLD_V3) ===")
print(r1_boot_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

# Phép so sánh chính của RQ2: ứng viên khác rf có macro-F1 trung bình cao nhất
non_ref_df = r1_summary_df[r1_summary_df["classifier"] != "rf"].sort_values("macro_f1_mean", ascending=False)
best_cand = str(non_ref_df.iloc[0]["classifier"])
primary_row = r1_boot_df[r1_boot_df["candidate"] == best_cand].iloc[0]
passes_rule = (float(primary_row["point_diff"]) >= 0.005) and (not bool(primary_row["contains_zero"]))
selected_classifier = best_cand if passes_rule else "rf"

# Đối chiếu chéo bổ sung với cấu hình Random Forest 500 cây của RQ1 (rq1_ghep_rf)
runs_cross = {
    "rq2_svm": {42: load_rq2_predictions(RESULTS_DIR, "ghep", "svm", 42)},
    "rq1_rf": {s: load_predictions(RESULTS_DIR, "ghep", s) for s in SEEDS_ORDER},
}
y_cross, preds_cross = align_predictions(runs_cross)
cross_res = paired_bootstrap_diff(y_cross, preds_cross["rq2_svm"], preds_cross["rq1_rf"], n_boot=2000, rng_seed=42)

print(f"\nPhép so sánh chính RQ2: '{best_cand}' vs 'rf' (rq2_ghep_rf)")
print(f"  Hiệu số Macro-F1 (point_diff) = {primary_row['point_diff']:+.6f} (Ngưỡng >= +0.005)")
print(f"  95% CI Bootstrap ghép cặp     = [{primary_row['ci_low']:+.6f}, {primary_row['ci_high']:+.6f}] (contains_zero = {primary_row['contains_zero']})")
print(f"Đối chiếu chéo bổ sung: '{best_cand}' vs 'rq1_ghep_rf' (500 cây cố định ở RQ1)")
print(f"  Hiệu số Macro-F1 (point_diff) = {cross_res['point_diff']:+.6f}, 95% CI = [{cross_res['ci_low']:+.6f}, {cross_res['ci_high']:+.6f}] (contains_zero = {cross_res['contains_zero']})")
print(f"==> QUYẾT ĐỊNH CHỐT BỘ PHÂN LOẠI TỐI ƯU NHẤT CHO RQ3 VÀ TEST CUỐI KỲ (S4-04): '{selected_classifier}'")


=== BẢNG R1: MACRO-F1 VÀ RECALL TỪNG LỚP CỦA 6 BỘ PHÂN LOẠI TRÊN NHÓM GHEP ===
classifier           macro_f1_report  ci95_low  ci95_high  recall_COVID_mean  recall_Lung_Opacity_mean  recall_Normal_mean  recall_Viral Pneumonia_mean
       svm  0.9024 (tất định, 1 run)    0.8909     0.9129             0.9572                    0.8494              0.8837                       0.9409
        gb 0.8868 ± 0.0030 (3 seeds)    0.8749     0.8976             0.9485                    0.8261              0.8707                       0.8949
        rf 0.8582 ± 0.0033 (3 seeds)    0.8432     0.8716             0.9025                    0.7697              0.9012                       0.7619
        lr  0.8370 (tất định, 1 run)    0.8221     0.8505             0.8678                    0.8306              0.8014                       0.9458
       knn  0.8242 (tất định, 1 run)    0.8075     0.8397             0.8194                    0.7464              0.8857                       0.7438
        d

## 4. Báo cáo bắt buộc R2 (Ma trận con `Normal – Lung_Opacity`), R4 (Sai số lấy mẫu) và R5 (Giới hạn)


In [5]:
r2_df = pd.read_csv(report_paths["r2_normal_lung_opacity.csv"])
r4_df = pd.read_csv(report_paths["r4_sampling_error.csv"])
r5_str = report_paths["r5_gioi_han.md"].read_text(encoding="utf-8").strip()

print("=== BẢNG R2: MA TRẬN CON 2x2 NORMAL - LUNG OPACITY (TRUNG BÌNH THEO BỘ PHÂN LOẠI) ===")
r2_mean = r2_df.groupby("classifier", as_index=False, sort=False)[["LO_to_LO", "LO_to_Normal", "Normal_to_LO", "Normal_to_Normal"]].mean()
print(r2_mean.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

print("\n=== BẢNG R4: SAI SỐ CHUẨN LẤY MẪU XẤP XỈ (TRUNG BÌNH THEO BỘ PHÂN LOẠI) ===")
r4_mean = r4_df.groupby("classifier", as_index=False, sort=False)[["p_accuracy", "se_all", "p_recall_viral", "se_viral"]].mean()
print(r4_mean.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== R5: GIỚI HẠN NGHIÊN CỨU ===")
print(r5_str)


=== BẢNG R2: MA TRẬN CON 2x2 NORMAL - LUNG OPACITY (TRUNG BÌNH THEO BỘ PHÂN LOẠI) ===
classifier  LO_to_LO  LO_to_Normal  Normal_to_LO  Normal_to_Normal
        lr     750.0         125.0         201.0            1227.0
       knn     674.0         211.0         157.0            1356.0
        dt     690.0         149.0         324.3            1092.0
        rf     695.0         201.7         136.7            1379.7
       svm     767.0         124.0         144.0            1353.0
        gb     746.0         150.0         167.7            1333.0

=== BẢNG R4: SAI SỐ CHUẨN LẤY MẪU XẤP XỈ (TRUNG BÌNH THEO BỘ PHÂN LOẠI) ===
classifier  p_accuracy  se_all  p_recall_viral  se_viral
        lr      0.8302  0.0067          0.9458    0.0159
       knn      0.8258  0.0067          0.7438    0.0306
        dt      0.7388  0.0078          0.7586    0.0300
        rf      0.8551  0.0062          0.7619    0.0299
       svm      0.8900  0.0056          0.9409    0.0166
        gb      0.8727  0.